# 05 — Exploração de dados: o que já temos?

Primeiro passo do projeto novo (risco / anomalias na Amazônia, ver `PLAN.md`): fazer um
**inventário dos dados que já existem no repositório**, sem depender do Earth Engine (o acesso
ainda está pendente).

Para cada arquivo: o que é, de onde veio, formato (shapes), e alguns exemplos visuais.

| # | Arquivo | Gerado por |
|---|---|---|
| 1 | `data/processed.zip` → `s2_patches.npz` | `notebooks/01_dados.ipynb` (projeto antigo, pastagem) |
| 2 | `experiments/amazon_anomaly_firstpass/data/amazon_firstpass.npz` | `experiments/amazon_anomaly_firstpass/build_dataset.py` |
| 3 | `model/grassland_embeddings_dataset.npz` | `model/grassland_quality_finetuning.ipynb` (projeto antigo) |

In [ ]:
import io
import zipfile
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from dataset_utils import rgb_composite

# Raiz do repositório (o notebook roda de dentro de notebooks/)
ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()


def carregar_npz(path_or_buffer):
    """Carrega um .npz inteiro como dict de arrays numpy."""
    d = np.load(path_or_buffer, allow_pickle=True)
    return {k: d[k] for k in d.files}


def resumo_npz(arrays):
    """Tabela com nome, shape e dtype de cada array."""
    return pd.DataFrame(
        [{'array': k, 'shape': v.shape, 'dtype': str(v.dtype)} for k, v in arrays.items()]
    ).set_index('array')

## 1. `data/processed.zip`: recortes Sentinel-2 (Goiânia, projeto antigo)

Este é o único arquivo com **imagens** de verdade: recortes de 64×64 pixels (10 m/pixel, ou seja
640 m × 640 m) com 6 bandas do Sentinel-2, em pontos de pastagem perto de Goiânia, rotulados com o
vigor da pastagem do MapBiomas. Lemos o `.npz` direto de dentro do zip, sem extrair para
`data/processed/` (essa pasta é a que os notebooks 02/03 leem).

In [ ]:
def carregar_patches_do_zip(zip_path=ROOT / 'data' / 'processed.zip'):
    with zipfile.ZipFile(zip_path) as z:
        return carregar_npz(io.BytesIO(z.read('processed/s2_patches.npz')))


patches = carregar_patches_do_zip()
resumo_npz(patches)

In [ ]:
print('Bandas:', patches['band_names'].tolist())
print('Rótulo (vigor MapBiomas -> classe):', patches['label_map'].item())
print('Anos:', np.unique(patches['vigor_year']).tolist())
print(f"Região: lat {patches['lat'].min():.2f} a {patches['lat'].max():.2f}, "
      f"lon {patches['lon'].min():.2f} a {patches['lon'].max():.2f}")

### Alguns exemplos (composição RGB = bandas B4, B3, B2)

In [ ]:
def mostrar_patches(X, idxs):
    fig, axes = plt.subplots(1, len(idxs), figsize=(2.0 * len(idxs), 2.3))
    for ax, i in zip(axes, idxs):
        ax.imshow(rgb_composite(X[i]))
        ax.set_title(f'recorte #{i}', fontsize=8)
        ax.axis('off')
    plt.tight_layout()
    plt.show()


rng = np.random.default_rng(42)
mostrar_patches(patches['X'], rng.choice(len(patches['X']), 8, replace=False))

Cada recorte aparece como um **quadrado de uma cor só**. Uma imagem real de 640 m de pastagem
teria textura (cercas, manchas de solo, árvores). Para confirmar, contamos quantos valores
diferentes existem dentro de cada recorte na banda B8. Uma imagem real teria centenas.

In [ ]:
X = patches['X']
n_unicos = np.array([len(np.unique(X[i, ..., 3])) for i in range(len(X))])
pd.Series(n_unicos).value_counts().rename_axis('valores diferentes em B8').to_frame('nº de recortes')

**Conclusão:** este zip contém a **versão antiga, com o bug** descrito no `README_pt.md` (seção
*Dados → Correção de tratamento*): a extração amostrava o Sentinel-2 numa grade grosseira e cada
recorte virava um único pixel replicado. A versão corrigida foi gerada depois, mas não foi
versionada (`data/processed/` está no `.gitignore`).

Mesmo corrigido, este dataset é de **pastagem em Goiás**, não da Amazônia. **Não serve como
imagem para o projeto novo.**

## 2. `amazon_firstpass.npz`: séries temporais na Amazônia (experimento do Gabriel)

O dado mais relevante para o projeto novo. **Não são imagens:** cada linha é um **ponto** (um pixel
de 10 m) na região do Tapajós / BR-163 (PA), e para cada ponto temos **7 anos** (2019 a 2025) de:

- `AE`: embedding AlphaEarth daquele pixel naquele ano (64 números) → tensor `(pontos, anos, 64)`
- `S2`: mediana anual do Sentinel-2 naquele pixel (6 bandas + NDVI + NBR) → tensor `(pontos, anos, 8)`

E o **rótulo** vem das transições do MapBiomas (`strat`), com o **ano do evento** `T`:

| `strat` | estrato | regra (MapBiomas) |
|---|---|---|
| 0 | floresta estável | floresta em T-2, T-1, T, T+1 |
| 1 | pastagem estável | pastagem em T-2, T-1, T, T+1 |
| 2 | desmatamento | floresta em T-2 e T-1 → uso humano em T e T+1 |
| 3 | mineração | floresta em T-2 e T-1 → mineração em T e T+1 |

Para os estratos estáveis, `T` é só o ano de referência em que o ponto foi sorteado.

In [ ]:
ESTRATOS = {0: 'floresta estável', 1: 'pastagem estável', 2: 'desmatamento', 3: 'mineração'}

amz = carregar_npz(ROOT / 'experiments' / 'amazon_anomaly_firstpass' / 'data' / 'amazon_firstpass.npz')
resumo_npz(amz)

In [ ]:
anos = amz['years']
print('Anos:', anos.tolist())
print('S2 (bandas/índices):', amz['s2_names'].tolist())
print('Valores ausentes (NaN): AE =', int(np.isnan(amz['AE']).sum()), '| S2 =', int(np.isnan(amz['S2']).sum()))
print('Partição (blocos espaciais de ~11 km):', pd.Series(amz['split']).value_counts().to_dict())

pd.crosstab(pd.Series(amz['strat']).map(ESTRATOS).rename('estrato'),
            pd.Series(amz['T']).rename('ano do evento T'), margins=True, margins_name='total')

### Onde estão os pontos

In [ ]:
fig, ax = plt.subplots(figsize=(6, 6))
for s, nome in ESTRATOS.items():
    m = amz['strat'] == s
    ax.scatter(amz['lon'][m], amz['lat'][m], s=4, label=nome)
ax.set_xlabel('longitude')
ax.set_ylabel('latitude')
ax.set_aspect('equal')
ax.set_title('Pontos amostrados: Tapajós / BR-163 (PA), ~220 × 220 km')
ax.legend(markerscale=3)
plt.show()

### Alguns exemplos: como um ponto evolui ao longo dos anos

Para 5 pontos de cada estrato com evento em 2022, duas séries:

- **NDVI** (do Sentinel-2): "quão verde" está o pixel. Floresta fica perto de 0,8–0,9.
- **Mudança do embedding**: distância cosseno entre o embedding AlphaEarth de um ano e o do ano
  anterior (0 = igual ao ano anterior). É o mesmo tipo de score de mudança que o Gabriel usou.

In [ ]:
def serie_s2(i, nome='NDVI'):
    return amz['S2'][i, :, amz['s2_names'].tolist().index(nome)]


def mudanca_embedding(i):
    # Distância cosseno entre anos consecutivos (o primeiro ano não tem anterior)
    e = amz['AE'][i] / np.linalg.norm(amz['AE'][i], axis=1, keepdims=True)
    return np.r_[np.nan, 1 - (e[1:] * e[:-1]).sum(axis=1)]


T_EXEMPLO = 2022
rng = np.random.default_rng(0)
fig, axes = plt.subplots(2, 4, figsize=(14, 5.5), sharex=True, sharey='row')
for col, (s, nome) in enumerate(ESTRATOS.items()):
    candidatos = np.where((amz['strat'] == s) & (amz['T'] == T_EXEMPLO))[0]
    for i in rng.choice(candidatos, 5, replace=False):
        axes[0, col].plot(anos, serie_s2(i), marker='o', alpha=0.7)
        axes[1, col].plot(anos, mudanca_embedding(i), marker='o', alpha=0.7)
    for ax in axes[:, col]:
        ax.axvline(T_EXEMPLO, color='k', ls='--', lw=1)
    axes[0, col].set_title(nome)
axes[0, 0].set_ylabel('NDVI')
axes[1, 0].set_ylabel('mudança do embedding\n(vs. ano anterior)')
fig.suptitle(f'5 pontos por estrato, evento em T = {T_EXEMPLO} (linha tracejada)')
plt.tight_layout()
plt.show()

## 3. `model/grassland_embeddings_dataset.npz`: embeddings AlphaEarth (Goiânia, projeto antigo)

300 pontos de pastagem perto de Goiânia, com o embedding AlphaEarth de 2018 a 2023 e o vigor da
pastagem como rótulo. Mesmo formato do item 2 (pontos × anos × 64), mas da região e do problema
antigos.

In [ ]:
grass = carregar_npz(ROOT / 'model' / 'grassland_embeddings_dataset.npz')
print('Anos:', grass['years'].tolist(), '| rótulos:', {int(k): v for k, v in grass['label_map'].item().items()})
resumo_npz(grass)

## Resumo: o que temos para trabalhar

| Arquivo | É imagem? | Região | Tempo | Rótulo | Serve pro projeto novo? |
|---|---|---|---|---|---|
| `processed.zip` | sim, mas **quebrada** (cor única) | Goiânia | 1 ano por ponto | vigor de pastagem | não |
| `amazon_firstpass.npz` | não: **pontos × anos** | **Tapajós (PA)** | **2019–2025** | estrato MapBiomas + ano do evento | **sim, é o principal** |
| `grassland_embeddings_dataset.npz` | não: pontos × anos | Goiânia | 2018–2023 | vigor de pastagem | não |

**O que não temos localmente:** imagens (recortes espaciais) da Amazônia.

## 4. Recortes da Amazônia baixados do Earth Engine

O resumo acima mostrou que não tínhamos **imagens** da Amazônia. As funções em `utils/` baixam
recortes do Earth Engine (parâmetros e detalhes no topo de `utils/download_ee.py`):

- **Região:** ecorregiões de floresta amazônica (RESOLVE 2017) dentro do Brasil.
- **Sorteio:** o critério só diz *onde olhar*, não é rótulo. Ele é checado no pixel do MapBiomas
  onde cai o ponto sorteado:
  - 6 `uniforme`: qualquer ponto da Amazônia, com chance proporcional à área;
  - 3 `mineracao`: o MapBiomas marca mineração (classe 30) em pelo menos um ano;
  - 3 `mudou`: o MapBiomas dá mais de uma classe entre 2019 e 2025.
- **Cada recorte:** 256 × 256 pixels de 10 m (2,56 km de lado), centrado no ponto sorteado, de 2019 a 2025.
  - `X`: composto anual do Sentinel-2 (mediana das passagens sem nuvem), 6 bandas → `(recortes, anos, 256, 256, 6)`
  - `mapbiomas`: classe do MapBiomas 10 m em cada pixel → `(recortes, anos, 256, 256)`

Os arquivos ficam em `data/raw/amazonia_exploracao/` (fora do git). Rodar a célula de novo não baixa outra vez.

In [ ]:
import sys
sys.path.append(str(ROOT))
from utils.download_ee import baixar_amostra_amazonia, carregar_amostra

baixar_amostra_amazonia(
    nome='amazonia_exploracao',
    n_por_criterio={'uniforme': 6, 'mineracao': 3, 'mudou': 3},
    anos=range(2019, 2026),
    tamanho_px=256,
    seed=42,
)
X, mapbiomas, anos_rec, info = carregar_amostra('amazonia_exploracao')
print('X:', X.shape, X.dtype, '| mapbiomas:', mapbiomas.shape, mapbiomas.dtype)
info

### Todas as imagens

Uma linha por recorte, uma coluna por ano. Composição RGB (bandas B4, B3, B2): cor natural. O `+`
vermelho marca o ponto sorteado. Preto = nenhuma observação sem nuvem naquele ano.

In [ ]:
def mostrar_recortes(X, info, anos, ganho=3.5):
    n, t, h, w = X.shape[:4]
    fig, axes = plt.subplots(n, t, figsize=(1.6 * t, 1.6 * n), squeeze=False)
    for i in range(n):
        for j in range(t):
            ax = axes[i, j]
            ax.imshow(rgb_composite(X[i, j], gain=ganho))
            ax.plot(w / 2, h / 2, '+', color='red', ms=7, mew=1)
            ax.set_xticks([])
            ax.set_yticks([])
            if i == 0:
                ax.set_title(anos[j])
        eco = info.ecorregiao[i].replace(' moist forests', '')
        axes[i, 0].set_ylabel(f'#{i} {info.criterio[i]}\n{eco}', fontsize=7)
    plt.tight_layout()
    plt.show()


mostrar_recortes(X, info, anos_rec)